In [ ]:
# Install Prereqs
%pip install -qU langchain-text-splitters
%pip install -qU langchain-ollama
%pip install langchain.
%pip install ollama  

In [11]:
%pip install langchain-chroma

  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
    --------------------------------------- 0.5/23.5 MB 2.8 MB/s eta 0:00:09
   -- ------------------------------------- 1.3/23.5 MB 3.5 MB/s eta 0:00:07
   --- ------------------------------------ 2.1/23.5 MB 3.9 MB/s eta 0:00:06
   ----- ---------------------------------- 3.4/23.5 MB 4.5 MB/s eta 0:00:05
   -------- ------------------------------- 4.7/23.5 MB 4.9 MB/s eta 0:00:04
   ----------- ---------------------------- 6.6/23.5 MB 5.6 MB/s eta 0:00:04
   -------------- ------------------------- 8.7/23.5 MB 6.3 MB/s eta 0:00:03
   ------------------- -------------------- 11.5/23.5 MB 7.4 MB/s eta 0:00:02
   ------------------------- -------------- 15.2/23.5 MB 8.5 MB/s eta 0:00:01
   -------------------------------- ------- 19.1/23.5 MB 9.7 MB/s eta 0:00:01
   ---------------------------------------  23.3/23.5 MB 10.8 MB/s eta 0:00:01
   ---------

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
roboflow 1.2.13 requires idna==3.7, but you have idna 3.20 which is incompatible.


In [12]:
# Step 1: Chunking

# I want to try Recursive Character Splitting via
# https://www.reddit.com/r/Rag/comments/1r47duk/we_benchmarked_7_chunking_strategies_most_best/

# https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

dataset = []

# TODO: Change these parameters later
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

# TODO: Be able to take in PDFs/other media files.
with os.scandir('sample-data') as es:
    for e in es:
        if e.is_file() and e.name.endswith('.md'):
            with open(e.path, 'r', encoding='utf-8') as file:
                file_metadata = {'filename': file.name} # We need the file name to source it later on.
                dataset.extend(text_splitter.create_documents(texts=[file.read()], metadatas=[file_metadata])) # Extend instead of append

dataset

[Document(metadata={'filename': 'sample-data\\advisory-group-members.md'}, page_content='---\nlayout: default\ntitle: Advisory Group Members\ndescription: "This page lists the members of the Diversity, Inclusion & Belonging Advisory Group."\n---\n\n{{< group-by-expertise "Diversity, Inclusion & Belonging Advisory Group" >}}'),
 Document(metadata={'filename': 'sample-data\\being-an-ally.md'}, page_content='---\ntitle: "The Ally Lab"\ndescription: Learn what is an ally, how to be an ally and what it means to be an ally.\n---\n\n## What is an ally?\n\nA diversity, inclusion and belonging "ally" is someone who is willing to take action in support of another person, in order to remove barriers that impede that person from contributing their skills and talents in the workplace or community.\n\nBeing an ally is a verb, this means that you proactively and purposefully take action and is not something forced upon you.'),
 Document(metadata={'filename': 'sample-data\\being-an-ally.md'}, page_con

In [ ]:
# Step 2: Embedding, Vector Database and Retrieval

import numpy as np
from langchain_ollama import OllamaEmbeddings

# https://reference.langchain.com/python/langchain-core/embeddings/embeddings/Embeddings
# https://docs.langchain.com/oss/python/integrations/embeddings/ollama 

# I'm using llama3 since its local
# TODO: Explore different embedding models for deployment
# ollama pull llama3
EMBEDDING_MODEL = 'llama3'

embeddings = OllamaEmbeddings (
    model=EMBEDDING_MODEL,
)

# from langchain_core.vectorstores import InMemoryVectorStore
from langchain_chroma.vectorstores import Chroma

# Vector database from all the documents embedded
# https://github.com/hwchase17/chroma-langchain/blob/master/persistent-qa.ipynb 
# TODO: Consider FAISS?

persist_directory = 'db'

vectorstore = Chroma.from_documents(
    documents=[dataset[i] for i in range(len(dataset))],
    embedding=embeddings,
    persist_directory=persist_directory,
)

vectorstore = None # Set to NULL to test we are pulling from the correct db

# print(vectorstore) # Debug

vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embeddings)

# Set vectorstore as the retriever
# https://reference.langchain.com/python/langchain-core/vectorstores/base/VectorStore/as_retriever
retriever = vectorstore.as_retriever()

# Retrieve the most similar text
retrieved_documents = retriever.invoke("What is an ally?")

print(retrieved_documents[0].page_content)
# That's not a bad result!

None
To be an effective ally it is important to understand some of the skills and behaviors great allies exhibit.


In [18]:
# Step 3: Generation

from langchain_ollama import OllamaLLM

# TODO: Change these parameters
model = OllamaLLM(
    model = 'llama3',
    temperature = 0.5, 
    num_predict = 256,
)

input_text = 'What is an ally?'
response = model.invoke(input_text)
print(response)
# Example without RAG prompting

An ally is someone who actively supports and stands in solidarity with a marginalized or oppressed group, often sharing their privilege and resources to help create a more just and equitable society. Allies can be individuals from a dominant group (e.g., white people, cisgender people) who use their positionality to advocate for and amplify the voices of marginalized groups.

Effective allies:

1. **Listen and learn**: Educate themselves about the issues facing the marginalized group, and listen to their experiences and concerns.
2. **Use their privilege**: Recognize their own advantages and use them to benefit the marginalized group. For example, using their voice to speak out against discrimination or advocating for policies that support the group.
3. **Amplify marginalized voices**: Give space and platform to the marginalized group's voices, rather than speaking over them or assuming they know what is best.
4. **Be accountable**: Be willing to confront and correct their own biases a

In [19]:
# Step 4: Prompting

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# TODO: Prompt adjustments
prompt = ChatPromptTemplate.from_template(""" Answer using ONLY the context below. If the context doesn't contain the answer,
then mention that it doesn't. Cite the source file from the metadata as ONLY a file name ONLY in a section of its own.

Context:
{context}

Question: {question}
""")

RAG = (
    {
    'context': retriever,
    'question': RunnablePassthrough()
    }
    | prompt
    | model
)

answer = RAG.invoke('What is an ally?')
print(answer)
# Looks pretty good!
# My biggest pet peave here is that it'll cite the source elsewhere as well- but I only want it to cite it at the end.

According to the context, an ally is someone who is willing to take action in support of another person, in order to remove barriers that impede that person from contributing their skills and talents in the workplace or community.

Source file: being-an-ally.md
